In [ ]:
# -*- coding: utf-8 -*-
"""NDWI maximum composite - with checkpointing and failed tile retry"""

!pip install rasterio oauthlib requests requests-oauthlib pyproj

from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
import numpy as np
import tarfile
import io
import os
import time
import random
import json
import rasterio
from rasterio.io import MemoryFile
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from google.colab import drive
from pyproj import Transformer

# -----------------------------
# 1. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

output_dir  = '/content/drive/MyDrive/RemoteSensingResearchRelated/ColabNotebooks/ndwi_10m_tiled'
tile_dir    = os.path.join(output_dir, "tiles")
failed_log  = os.path.join(output_dir, "failed_tiles.json")   # tracks failed tiles
os.makedirs(output_dir, exist_ok=True)
os.makedirs(tile_dir,   exist_ok=True)

# -----------------------------
# 2. Credentials
# -----------------------------
client_id     = "YOUR_COPERNICUS_CLIENT_ID"
client_secret = "YOUR_COPERNICUS_CLIENT_SECRET"

# -----------------------------
# 3. Authentication
# -----------------------------
def getauth_token():
    client = BackendApplicationClient(client_id=client_id)
    oauth  = OAuth2Session(client=client)
    token  = oauth.fetch_token(
        token_url="https://identity.cloudferro.com/auth/realms/CDSE/protocol/openid-connect/token",
        client_id=client_id,
        client_secret=client_secret,
    )
    return oauth, token

def ensure_valid_oauth(oauth, token, buffer_seconds=120):
    now        = time.time()
    expires_at = token.get("expires_at", None)
    if expires_at is None or now >= (expires_at - buffer_seconds):
        print("Token expired. Refreshing...")
        oauth, token = getauth_token()
    return oauth, token

# -----------------------------
# 4. Time slots
# -----------------------------
slots = []
for month in ['01', '02', '03', '04', '10', '11', '12']:
    month_group = [
        (f"2021-{month}-01", f"2021-{month}-28"),
        (f"2022-{month}-01", f"2022-{month}-28"),
        (f"2023-{month}-01", f"2023-{month}-28"),
        (f"2024-{month}-01", f"2024-{month}-28"),
    ]
    slots.append(month_group)

# -----------------------------
# 5. Evalscript
# -----------------------------
evalscript = """
function setup() {
  return {
    input: ["B03", "B08", "dataMask"],
    output: { id: "default", bands: 3, sampleType: "FLOAT32" }
  };
}
function evaluatePixel(sample) {
  return [sample.B03, sample.B08, sample.dataMask];
}
"""

# -----------------------------
# 6. AOI + canvas setup
# -----------------------------
lon_min, lat_min = 81.45785785, 29.06047142
lon_max, lat_max = 82.04838275, 29.40491605

transformer      = Transformer.from_crs("EPSG:4326", "EPSG:32644", always_xy=True)
x_min, y_min     = transformer.transform(lon_min, lat_min)
x_max, y_max     = transformer.transform(lon_max, lat_max)
TARGET_RES       = 10

canvas_cols      = int(round((x_max - x_min) / TARGET_RES))
canvas_rows      = int(round((y_max - y_min) / TARGET_RES))
full_transform   = from_bounds(x_min, y_min, x_max, y_max, canvas_cols, canvas_rows)
full_crs         = CRS.from_epsg(32644)

print(f"Full AOI canvas: {canvas_rows} rows x {canvas_cols} cols at {TARGET_RES}m")

base_profile = {
    "driver"    : "GTiff",
    "dtype"     : "float32",
    "width"     : canvas_cols,
    "height"    : canvas_rows,
    "count"     : 1,
    "crs"       : full_crs,
    "transform" : full_transform,
    "nodata"    : np.nan,
    "compress"  : "lzw",
    "tiled"     : True,
    "blockxsize": 256,
    "blockysize": 256,
}

# -----------------------------
# 7. Tiling
# -----------------------------
tile_size_m = 20000

def create_tiles(xmin, ymin, xmax, ymax, tile_size):
    tiles = []
    x = xmin
    while x < xmax:
        x2 = min(x + tile_size, xmax)
        y  = ymin
        while y < ymax:
            y2 = min(y + tile_size, ymax)
            tiles.append([x, y, x2, y2])
            y = y2
        x = x2
    return tiles

tiles = create_tiles(x_min, y_min, x_max, y_max, tile_size_m)
print(f"Number of tiles: {len(tiles)}")

# -----------------------------
# 8. Canvas helpers
# -----------------------------
def bbox_to_pixel_window(bbox, res):
    col_off = int(round((bbox[0] - x_min) / res))
    row_off = int(round((y_max  - bbox[3]) / res))
    width   = int(round((bbox[2] - bbox[0]) / res))
    height  = int(round((bbox[3] - bbox[1]) / res))
    return row_off, col_off, height, width

def nan_safe_max_canvas(canvas, patch, row_off, col_off, height, width):
    height = min(height, canvas.shape[0] - row_off)
    width  = min(width,  canvas.shape[1] - col_off)
    patch  = patch[:height, :width]
    region = canvas[row_off:row_off+height, col_off:col_off+width]
    canvas[row_off:row_off+height, col_off:col_off+width] = np.where(
        np.isnan(region), patch,
        np.where(np.isnan(patch), region, np.maximum(region, patch))
    )
    return canvas

# -----------------------------
# 9. NDWI
# -----------------------------
def calculate_ndwi(green, nir, mask):
    green = green.astype(np.float32)
    nir   = nir.astype(np.float32)
    mask  = mask.astype(np.float32)
    ndwi  = np.full(green.shape, np.nan, dtype=np.float32)
    valid = (mask == 1) & ((green + nir) != 0)
    ndwi[valid] = (green[valid] - nir[valid]) / (green[valid] + nir[valid])
    return ndwi

# -----------------------------
# 10. Request with retry + longer timeout for cloudy months
# -----------------------------
CLOUDY_MONTHS = ['03', '04', '10', '11', '12']  # months prone to timeouts

def get_tile_request(slot, bbox_utm, oauth, token, max_retries=6):
    oauth, token = ensure_valid_oauth(oauth, token)

    # Use weekly sub-slots for cloudy months to reduce server load per request
    month = slot[0][5:7]
    if month in CLOUDY_MONTHS:
        # Split the month into 2-week windows — smaller time range = faster server
        sub_slots = [
            (slot[0], slot[0][:8] + "14"),   # day 1-14
            (slot[0][:8] + "15", slot[1]),    # day 15-28
        ]
    else:
        sub_slots = [slot]

    # Try each sub-slot
    for sub_slot in sub_slots:
        request = {
            "input": {
                "bounds": {
                    "bbox": bbox_utm,
                    "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/32644"}
                },
                "data": [{
                    "type": "sentinel-2-l2a",
                    "dataFilter": {
                        "timeRange": {
                            "from": sub_slot[0] + "T00:00:00Z",
                            "to":   sub_slot[1] + "T23:59:59Z"
                        },
                        "maxCloudCoverage": 15
                    }
                }]
            },
            "output": {
                "resx": TARGET_RES,
                "resy": TARGET_RES,
                "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
            },
            "evalscript": evalscript
        }

        url = "https://sh.dataspace.copernicus.eu/api/v1/process"

        for attempt in range(1, max_retries + 1):
            try:
                oauth, token = ensure_valid_oauth(oauth, token)
                response     = oauth.post(url, json=request, headers={"Accept": "application/tar"})

                if response.status_code == 200:
                    return response, oauth, token

                elif response.status_code in [500, 502, 503]:
                    wait = 2 ** attempt + random.uniform(1, 3)
                    print(f"\n    Attempt {attempt}/{max_retries} "
                          f"(HTTP {response.status_code}). Retrying in {wait:.1f}s...")
                    time.sleep(wait)

                else:
                    raise RuntimeError(f"Non-retryable error {response.status_code}: {response.text}")

            except RuntimeError:
                raise
            except Exception as e:
                wait = 2 ** attempt + random.uniform(1, 3)
                print(f"\n    Connection error: {e}. Retrying in {wait:.1f}s...")
                time.sleep(wait)

    raise RuntimeError(f"All {max_retries} attempts failed.")

# -----------------------------
# 11. Failed tile log helpers
# -----------------------------
def load_failed_log():
    if os.path.exists(failed_log):
        with open(failed_log, 'r') as f:
            return json.load(f)
    return {}

def save_failed_log(log):
    with open(failed_log, 'w') as f:
        json.dump(log, f, indent=2)

def log_failed_tile(log, month, slot, tile_idx):
    key = f"{month}_{slot[0]}_{slot[1]}"
    if key not in log:
        log[key] = []
    if tile_idx not in log[key]:
        log[key].append(tile_idx)
    save_failed_log(log)

def clear_failed_tile(log, month, slot, tile_idx):
    key = f"{month}_{slot[0]}_{slot[1]}"
    if key in log and tile_idx in log[key]:
        log[key].remove(tile_idx)
    save_failed_log(log)

# -----------------------------
# 12. Process one date
# -----------------------------
def process_one_date(slot, oauth, token, tiles, month, failed_log_data):
    tile_results = []

    for idx, tile_bbox in enumerate(tiles):
        print(f"  Tile {idx+1}/{len(tiles)}", end=" ")
        try:
            resp, oauth, token = get_tile_request(slot, tile_bbox, oauth, token)
            tar_stream = io.BytesIO(resp.content)

            with tarfile.open(fileobj=tar_stream, mode='r:*') as tar:
                tif_member = next((m for m in tar.getmembers() if m.name.endswith('.tif')), None)
                if tif_member is None:
                    print("→ No TIF, skipping.")
                    log_failed_tile(failed_log_data, month, slot, idx)
                    continue
                tif_bytes = tar.extractfile(tif_member).read()

            with MemoryFile(tif_bytes) as memfile:
                with memfile.open() as dataset:
                    green = dataset.read(1)
                    nir   = dataset.read(2)
                    mask  = dataset.read(3)
                    ndwi  = calculate_ndwi(green, nir, mask)
                    print(f"→ shape {ndwi.shape} ✓")
                    tile_results.append((ndwi, tile_bbox))
                    # Clear from failed log if it was previously failed
                    clear_failed_tile(failed_log_data, month, slot, idx)

        except Exception as e:
            print(f"→ FAILED: {e}, skipping.")
            log_failed_tile(failed_log_data, month, slot, idx)
            continue

    return tile_results, oauth, token

# -----------------------------
# 13. Main processing WITH checkpoint
# -----------------------------
oauth, token     = getauth_token()
failed_log_data  = load_failed_log()

absolute_max_canvas = np.full((canvas_rows, canvas_cols), np.nan, dtype=np.float32)

for month_group in slots:
    month_name  = month_group[0][0][5:7]
    month_path  = os.path.join(output_dir, f"max_ndwi_month_{month_name}.tif")

    # CHECKPOINT: Skip month if already saved successfully
    if os.path.exists(month_path):
        print(f"\n=== Month {month_name}: already saved, loading for absolute max ===")
        with rasterio.open(month_path) as src:
            saved = src.read(1)
        # Still contribute to absolute max
        absolute_max_canvas = np.where(
            np.isnan(absolute_max_canvas), saved,
            np.where(np.isnan(saved), absolute_max_canvas,
                     np.maximum(absolute_max_canvas, saved))
        )
        continue

    print(f"\n=== Processing Month: {month_name} ===")
    month_max_canvas = np.full((canvas_rows, canvas_cols), np.nan, dtype=np.float32)

    for slot in month_group:
        print(f"  Date slot: {slot[0]} to {slot[1]}")
        tile_results, oauth, token = process_one_date(
            slot, oauth, token, tiles, month_name, failed_log_data
        )
        for ndwi_patch, bbox in tile_results:
            row_off, col_off, height, width = bbox_to_pixel_window(bbox, TARGET_RES)
            month_max_canvas = nan_safe_max_canvas(
                month_max_canvas, ndwi_patch, row_off, col_off, height, width
            )

    # Save month
    with rasterio.open(month_path, "w", **base_profile) as dst:
        dst.write(month_max_canvas, 1)
    print(f"Saved: {month_path}")

    absolute_max_canvas = np.where(
        np.isnan(absolute_max_canvas), month_max_canvas,
        np.where(np.isnan(month_max_canvas), absolute_max_canvas,
                 np.maximum(absolute_max_canvas, month_max_canvas))
    )

# -----------------------------
# 14. Save absolute max
# -----------------------------
final_path = os.path.join(output_dir, "absolute_max_ndwi_10m.tif")
with rasterio.open(final_path, "w", **base_profile) as dst:
    dst.write(absolute_max_canvas, 1)

print(f"\nDone! Final output: {final_path}")
if failed_log_data:
    print(f"Some tiles still failed. Check: {failed_log}")
else:
    print("All tiles processed successfully!")